Install Libraries & Import Modules

In [11]:
# ============================================================
# Cell 1 : Install Required Libraries
# ============================================================

!pip -q install groq gradio pypdf pandas scikit-learn

# ============================================================
# Import Libraries
# ============================================================

import os
import json
import time
import warnings

import pandas as pd
import gradio as gr

from groq import Groq
from pypdf import PdfReader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from google.colab import files
from google.colab import userdata

warnings.filterwarnings("ignore")

print("=" * 60)
print("FoodExpress AI Chatbot Evaluation")
print("=" * 60)
print("✅ Libraries installed successfully")
print("✅ Modules imported successfully")
print("=" * 60)

FoodExpress AI Chatbot Evaluation
✅ Libraries installed successfully
✅ Modules imported successfully


Configure Groq API & Test Connection

In [12]:
# ============================================================
# Cell 2 : Configure Groq API
# ============================================================

from google.colab import userdata
from groq import Groq

# Read API Key from Colab Secrets
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    raise Exception(
        "❌ GROQ_API_KEY not found.\n"
        "Go to Colab → Secrets (🔑) → Add a secret named GROQ_API_KEY."
    )

if not GROQ_API_KEY:
    raise Exception("❌ GROQ_API_KEY is empty.")

# Create Groq Client
client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq Client Created Successfully")

# ============================================================
# Test API Connection
# ============================================================

try:

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": "Reply with only the word OK"
            }
        ],
        temperature=0
    )

    print("=" * 60)
    print("Groq API Test")
    print("=" * 60)
    print(response.choices[0].message.content)
    print("=" * 60)
    print("✅ API Connected Successfully")

except Exception as e:

    print("❌ API Connection Failed")
    print(e)

✅ Groq Client Created Successfully
Groq API Test
OK
✅ API Connected Successfully


Upload Restaurant PDF & Extract Knowledge

In [13]:
# ============================================================
# Cell 3 : Upload Restaurant PDF and Extract Knowledge
# ============================================================

from google.colab import files
from pypdf import PdfReader

print("=" * 60)
print("Upload Restaurant PDF")
print("=" * 60)

uploaded = files.upload()

restaurant_context = ""

pdf_count = 0

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        pdf_count += 1

        print(f"\nReading : {filename}")

        reader = PdfReader(filename)

        for page in reader.pages:

            text = page.extract_text()

            if text:
                restaurant_context += text + "\n"

if pdf_count == 0:
    raise Exception("❌ No PDF uploaded.")

# Clean extracted text
restaurant_context = " ".join(restaurant_context.split())

print("\n" + "=" * 60)
print("Restaurant Knowledge Loaded Successfully")
print("=" * 60)

print(f"PDF Files Loaded        : {pdf_count}")
print(f"Characters Extracted    : {len(restaurant_context)}")
print(f"Words Extracted         : {len(restaurant_context.split())}")

print("=" * 60)
print("Preview (First 500 Characters)")
print("=" * 60)

print(restaurant_context[:500])

print("\n✅ Restaurant Knowledge Base Ready")

Upload Restaurant PDF


Saving about_us.pdf to about_us (1).pdf
Saving restaurant_policy.pdf to restaurant_policy.pdf
Saving faq.pdf to faq.pdf
Saving offers.pdf to offers.pdf
Saving restaurant_menu.pdf to restaurant_menu.pdf

Reading : about_us (1).pdf

Reading : restaurant_policy.pdf

Reading : faq.pdf

Reading : offers.pdf

Reading : restaurant_menu.pdf

Restaurant Knowledge Loaded Successfully
PDF Files Loaded        : 5
Characters Extracted    : 2275
Words Extracted         : 301
Preview (First 500 Characters)
About FoodExpress FoodExpress is an AI-powered online restaurant ordering platform. We serve: • Pizza • Burgers • Pasta • Sandwiches • Desserts • Drinks Our Mission To deliver fresh, delicious, and affordable food quickly while providing excellent customer service. Business Hours 10:00 AM – 11:00 PM Support Email support@foodexpress.com Website www.foodexpress.com FoodExpress Restaurant Policies 1. Orders can be cancelled within 5 minutes. 2. Refunds are available only for damaged or incorrect o

✅

Upload Evaluation Dataset (CSV)

In [14]:
# ============================================================
# Cell 4 : Upload Evaluation Dataset (CSV)
# ============================================================

from google.colab import files
import pandas as pd

print("=" * 60)
print("Upload Evaluation CSV")
print("=" * 60)

uploaded = files.upload()

csv_file = None

for filename in uploaded.keys():
    if filename.lower().endswith(".csv"):
        csv_file = filename
        break

if csv_file is None:
    raise Exception("❌ Please upload a CSV file.")

# Read CSV
df = pd.read_csv(csv_file)

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Required columns
required_columns = ["Question", "Expected", "Predicted"]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise Exception(f"❌ Missing column(s): {missing}")

# Clean text columns
for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

print("\n" + "=" * 60)
print("Evaluation Dataset Loaded Successfully")
print("=" * 60)

print(f"Dataset File : {csv_file}")
print(f"Total Records: {len(df)}")

print("\nColumns:")
print(df.columns.tolist())

print("\nSample Records:")
display(df.head())

print("✅ Evaluation Dataset Ready")

Upload Evaluation CSV


Saving evaluation_results.csv to evaluation_results.csv

Evaluation Dataset Loaded Successfully
Dataset File : evaluation_results.csv
Total Records: 30

Columns:
['Question', 'Expected', 'Predicted']

Sample Records:


,Question,Expected,Predicted
0,What is FoodExpress?,FoodExpress is an AI-powered online restaurant...,FoodExpress is an AI-powered online restaurant...
1,What type of platform is FoodExpress?,AI-powered online restaurant ordering platform.,AI-powered online restaurant ordering platform.
2,What foods are available at FoodExpress?,"Pizza, Burgers, Pasta, Sandwiches, Desserts, a...","Pizza, Burgers, Pasta, Sandwiches, Desserts, a..."
3,What is the mission of FoodExpress?,"To deliver fresh, delicious, and affordable fo...","To deliver fresh, delicious, and affordable fo..."
4,What are the business hours of FoodExpress?,10:00 AM – 11:00 PM.,10:00 AM – 11:00 PM.


✅ Evaluation Dataset Ready


Initialize Evaluation Variables

In [15]:
# ============================================================
# Cell 5 : Initialize Evaluation Variables
# ============================================================

# Question Statistics
asked_questions = 0
correct_answers = 0

# Lists for Metrics
y_true = []
y_pred = []

# Performance Metrics
overall_accuracy = 0.0
precision = 0.0
recall = 0.0
f1 = 0.0

# Timing & Confidence
response_times = []
confidence_scores = []

# Chat History
chat_history = []

# Evaluation History
evaluation_history = []

print("=" * 60)
print("Evaluation Variables Initialized")
print("=" * 60)

print(f"Questions Asked      : {asked_questions}")
print(f"Correct Answers      : {correct_answers}")
print(f"Accuracy             : {overall_accuracy:.2f}%")
print(f"Precision            : {precision:.2f}")
print(f"Recall               : {recall:.2f}")
print(f"F1 Score             : {f1:.2f}")

print("=" * 60)
print("✅ Evaluation Engine Ready")
print("=" * 60)

Evaluation Variables Initialized
Questions Asked      : 0
Correct Answers      : 0
Accuracy             : 0.00%
Precision            : 0.00
Recall               : 0.00
F1 Score             : 0.00
✅ Evaluation Engine Ready


Semantic Question Matching & Chatbot Response

LLM Evaluation, Metrics Update & Return Results

In [18]:
# ============================================================
# Cell 6A : Semantic Question Matching + Chatbot Response
# ============================================================

def evaluate_chatbot(user_question):

    global restaurant_context
    global df

    global asked_questions
    global correct_answers

    global y_true
    global y_pred

    global overall_accuracy
    global precision
    global recall
    global f1

    global response_times
    global confidence_scores

    global chat_history
    global evaluation_history

    start_time = time.time()

    # --------------------------------------------------------
    # Build Question Bank
    # --------------------------------------------------------

    question_bank = ""

    for i, row in df.iterrows():

        question_bank += f"""
Question {i+1}
Question : {row['Question']}
Expected : {row['Expected']}

"""

    # --------------------------------------------------------
    # Ask LLM to Find Best Matching Question
    # --------------------------------------------------------

    matching_prompt = f"""
You are an intelligent evaluator.

A user asked:

"{user_question}"

Below is the evaluation dataset.

{question_bank}

Find the SINGLE BEST matching question.

Return ONLY valid JSON.

{{
"matched_question":"",
"expected_answer":""
}}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "user",
                "content": matching_prompt
            }
        ]
    )

    match_result = json.loads(
        response.choices[0].message.content
    )

    matched_question = match_result["matched_question"]
    expected_answer = match_result["expected_answer"]

    # --------------------------------------------------------
    # Generate Restaurant Chatbot Answer
    # --------------------------------------------------------

    chatbot_prompt = f"""
You are FoodExpress Restaurant AI Assistant.

Answer ONLY using the restaurant knowledge below.

If the answer is unavailable,
reply politely that the information is not available.

Restaurant Knowledge:

{restaurant_context}

Customer Question:

{user_question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        messages=[
            {
                "role": "user",
                "content": chatbot_prompt
            }
        ]
    )

    chatbot_answer = response.choices[0].message.content.strip()

        # --------------------------------------------------------
    # Evaluate Chatbot Answer using LLM
    # --------------------------------------------------------

    evaluation_prompt = f"""
You are an AI evaluator.

Compare the chatbot answer with the expected answer.

Expected Answer:
{expected_answer}

Chatbot Answer:
{chatbot_answer}

Judge semantic meaning, not exact wording.

Return ONLY valid JSON.

{{
    "correct": true,
    "confidence": 95,
    "reason": "Brief explanation"
}}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    )

    result = json.loads(
        response.choices[0].message.content
    )

    is_correct = bool(result["correct"])
    confidence = float(result["confidence"])
    reason = result["reason"]

    # --------------------------------------------------------
    # Update Statistics
    # --------------------------------------------------------

    asked_questions += 1

    if is_correct:
        correct_answers += 1

    y_true.append(1)
    y_pred.append(1 if is_correct else 0)

    overall_accuracy = accuracy_score(
        y_true,
        y_pred
    ) * 100

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    elapsed = time.time() - start_time

    response_times.append(elapsed)
    confidence_scores.append(confidence)

    # --------------------------------------------------------
    # Store Chat History
    # --------------------------------------------------------

    chat_history.append(
    {
        "role": "user",
        "content": user_question
    }
)

    chat_history.append(
    {
        "role": "assistant",
        "content": chatbot_answer
    }
)


    evaluation_history.append(
        {
            "Question": user_question,
            "Matched Question": matched_question,
            "Expected": expected_answer,
            "Predicted": chatbot_answer,
            "Correct": is_correct,
            "Confidence": confidence,
            "Reason": reason,
            "Response Time": round(elapsed, 2)
        }
    )

    # --------------------------------------------------------
    # Return Results
    # --------------------------------------------------------

    return (
    chat_history,
    f"{overall_accuracy:.2f} %",
    f"{precision:.2f}",
    f"{recall:.2f}",
    f"{f1:.2f}",
    f"{elapsed:.2f} sec",
    f"{confidence:.2f} %",
    matched_question,
    expected_answer,
    reason,
    chat_history
)

In [19]:
!pip install -U gradio

Gradio User Interface

In [ ]:
# ============================================================
# Cell 7 : FoodExpress AI Chatbot UI (Colab Compatible)
# ============================================================

import gradio as gr


def clear_all():
    return (
        [],      # chatbot
        "",      # question
        "",      # accuracy
        "",      # precision
        "",      # recall
        "",      # f1
        "",      # response time
        "",      # confidence
        "",      # matched question
        "",      # expected answer
        ""       # reason
    )


with gr.Blocks(
    title="FoodExpress AI Chatbot",
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown(
        """
# 🍽️ FoodExpress Restaurant AI Chatbot

Ask any restaurant-related question.

The chatbot answers using the uploaded Restaurant PDF and automatically evaluates
its response using the evaluation dataset and LLM.
"""
    )

    chatbot = gr.Chatbot(
        label="Conversation"
    )

    question = gr.Textbox(
        label="Your Question",
        placeholder="Example: What is today's special dish?",
        lines=2
    )

    with gr.Row():

        ask_btn = gr.Button(
            "Ask",
            variant="primary"
        )

        clear_btn = gr.Button("🗑 Clear")

    gr.Markdown("## 📊 Evaluation Metrics")

    with gr.Row():

        accuracy = gr.Textbox(label="Accuracy")

        precision = gr.Textbox(label="Precision")

        recall = gr.Textbox(label="Recall")

        f1 = gr.Textbox(label="F1 Score")

    with gr.Row():

        response_time = gr.Textbox(label="Response Time")

        confidence = gr.Textbox(label="Confidence")

    matched_question = gr.Textbox(
        label="Matched Evaluation Question",
        lines=2
    )

    expected_answer = gr.Textbox(
        label="Expected Answer",
        lines=4
    )

    reason = gr.Textbox(
        label="Evaluation Reason",
        lines=4
    )

    outputs = [
    chatbot,
    accuracy,
    precision,
    recall,
    f1,
    response_time,
    confidence,
    matched_question,
    expected_answer,
    reason
]


    ask_btn.click(
        fn=evaluate_chatbot,
        inputs=question,
        outputs=outputs
    )

    question.submit(
        fn=evaluate_chatbot,
        inputs=question,
        outputs=outputs
    )

    clear_btn.click(
        fn=clear_all,
        outputs=[
            chatbot,
            question,
            accuracy,
            precision,
            recall,
            f1,
            response_time,
            confidence,
            matched_question,
            expected_answer,
            reason
        ]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2b485d771bc3ce703f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
